In [ ]:
import os
import time
import requests
from llama_index.core import (
    SimpleDirectoryReader, 
    Settings, 
    StorageContext, 
    TreeIndex, 
    load_index_from_storage,
    Document
)
from llama_index.core.prompts.prompts import SimpleInputPrompt
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

class LlamaIndexTreeRAG:
    def __init__(self, model_name="gpt-oss:20b", embed_model_name="bge-m3"):
        self.model_name = model_name
        self.embed_model_name = embed_model_name
        self.index = None
        self.query_engine = None
        self.check_ollama_service()
        self.configure_settings()
        self.qa_template = SimpleInputPrompt("""
請根據提供的參考資料回答用戶問題。

【參考資料】        
---------------------
{context_str}
---------------------
問題：{query_str}

【指令規範】
1. **全面提取**：若資料包含答案，請以繁體中文清楚、完整地列出不得遺漏。
2. **忠於原文**：保持客觀，絕對不能因總結而遺漏任何資訊，優先使用資料內的用語。
3. **誠實原則**：如果資料中確實完全沒有提到相關資訊，請禮貌地回答「根據目前的資料庫，無法提供相關資訊」。
4. **完整性**：忽略資料中不相關的雜訊（如：分頁標記、無意義的編號）。
5. **後驗檢查**：在輸出答案前，請再次確認是否遺漏了【參考資料】中任何符合問題條件的細節。
        """)

    def check_ollama_service(self):
        """檢查 Ollama 伺服器是否正常運行"""
        ollama_url = "http://localhost:11434"
        try:
            response = requests.get(ollama_url)
            if response.status_code == 200:
                print(f"✅ Ollama 服務器 {ollama_url} 可達。")
            else:
                print(f"⚠️ Ollama 服務器回應異常，狀態碼: {response.status_code}")
        except requests.exceptions.RequestException:
            print(f"❌ 無法連接到 Ollama 服務器 ({ollama_url})，請確認服務已啟動。")

    def configure_settings(self):
        """配置 LlamaIndex 的 LLM 與 Embedding 模型，完全遵循 W11_1 的 chunk 設定"""
        Settings.llm = Ollama(model=self.model_name, request_timeout=30000)
        Settings.embed_model = OllamaEmbedding(model_name=self.embed_model_name)
        # 嚴格保留 W11_1 設定，絕不刪除修改
        Settings.chunk_size = 4096
        Settings.chunk_overlap = 512

    def load_existing_index(self, index_dir):
        """直接載入現有的儲存索引（比照 W10_1 載入邏輯）"""
        storage_context = StorageContext.from_defaults(persist_dir=index_dir)
        self.index = load_index_from_storage(storage_context)
        self.query_engine = self.index.as_query_engine()
        print(f"✅ 成功載入現有索引於 '{index_dir}'。")

    def _print_node_recursive(self, node_id: str, index_struct, docstore, depth: int, max_depth: int):
        """
        遞迴地列印節點及其子節點。
        """
        if depth > max_depth: return            
        indent = "  " * depth     
        try:
            node = docstore.get_document(node_id)
            if node is None:
                print(f"{indent}❌ 錯誤：節點 ID {node_id} 不存在於 docstore 中。")
                return
            summary = node.get_content()[:100].replace("\n", " ") + "..."
            print(f"{indent}└─ Node ID: {node.node_id} (Depth: {depth}) - Summary: {summary}")

            # 這是關鍵修改：從 index_struct 中獲取子節點 ID, 這樣可以確保我們只在有子節點關係的節點上進行遞迴
            child_node_ids = index_struct.get_children(node)
            if child_node_ids:
                for child_key, child_id in child_node_ids.items():
                    self._print_node_recursive(child_id, index_struct, docstore, depth + 1, max_depth)
        except Exception as e:
            print(f"{indent}❌ 錯誤：處理節點 {node_id} 時發生異常。原因: {e}")
            
    def create_index_from_documents(self, documents, index_dir):
        """建立新的 TreeIndex 並持久化儲存"""
        start_time = time.time()
        
        # 建立 TreeIndex 樹狀索引
        self.index = TreeIndex.from_documents(documents)
        
        # 將索引持久化儲存到指定資料夾
        self.index.storage_context.persist(persist_dir=index_dir)
        
        elapsed_time = time.time() - start_time
        print(f"✅ 已成功建立 TreeIndex 並儲存於 '{index_dir}'。")
        print(f"⏱️ 總共花費時間：{elapsed_time:.2f} 秒。")
        
        # 初始化查詢引擎
        self.query_engine = self.index.as_query_engine()

    def print_tree_structure_recursive(self, max_depth=3):
        """
        列印 TreeIndex 的樹狀架構，從根節點開始遞迴遍歷。        
        Args:
            max_depth: 樹狀結構的最大列印深度，避免輸出過長。
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
            
        print("\n🌳 TreeIndex 架構 (遞迴遍歷):")
        print("=" * 20)
        index_struct = self.index.index_struct
        docstore = self.index.docstore

        if not index_struct.root_nodes:
            print("❌ 錯誤：索引結構中沒有根節點。")
            return
        
        # 遍歷所有根節點並開始遞迴列印
        for root_key, root_id in index_struct.root_nodes.items():
            self._print_node_recursive(root_id, index_struct, docstore, depth=0, max_depth=max_depth)
        
        print("=" * 20)
    
    def print_tree_index_structure(self):
        """
        列印 TreeIndex 的樹狀架構。此方法會使用更穩健的方式來遍歷並列印節點。        
        Args:
            max_depth: 樹狀結構的最大列印深度，避免輸出過長。
        """
        if self.index is None:
            print("❌ 錯誤：索引尚未建立。")
            return
        print("\n🌳 TreeIndex 架構:\n","=" * 20)
        
        all_nodes = list(self.index.docstore.docs.values())  # 直接從 docstore 中獲取所有節點。      
        if not all_nodes:
            print("❌ 錯誤：docstore 中沒有找到任何節點。")
            return
        print(f"共有 {len(all_nodes)} 個節點。\n", "列印所有節點的摘要:")        
       
        for node in all_nodes:                           # 遍歷並列印每個節點的摘要
            summary = node.get_content()[:250].replace("\n", " ") + "..."    
            # 檢查元資料是否有檔案名稱和頁碼，是否有父節點
            file_name = node.metadata.get('file_name', 'N/A')
            page_label = node.metadata.get('page_label', 'N/A')
            parent_node_id = node.relationships.get('PARENT', 'N/A')            
            print("-" * 50)
            print(f"Node ID: {node.node_id}")
            print(f"來源檔案: {file_name}, 頁碼: {page_label}")
            print(f"父節點ID: {parent_node_id}")
            print(f"摘要: {summary}")           
        print("=" * 20)

    def rag_query(self, query):
            """
            對索引進行查詢並獲取 RAG 回答，並顯示相關的來源節點文本。            
            Args:
                query: 查詢字串                
            Returns:
                模型的回答
            """
            if self.index is None: return "錯誤：索引尚未建立，無法進行查詢。"
            
            print("🔍 正在查詢...")            
            # 建立一個查詢引擎，並傳入自訂的提示詞
            query_engine = self.index.as_query_engine(
                text_qa_template=self.qa_template,
                child_branch_factor=3  # 每一層選出 2 個最相關的分支往下找
            )
            
            try:
                response = query_engine.query(query)          # 執行查詢
                print(f"🤖 回答: {response.response}\n---\n") # 顯示最終回答
                print("📄 來源文件 (Source Nodes):")           # 顯示相關的來源節點文本
                if response.source_nodes:
                    for i, node in enumerate(response.source_nodes):
                        node_content = node.get_content().strip()
                        print(f"[{i+1}] Node ID: {node.node_id}")
                        print(f"內容摘要: {node_content[:200]}...") # 只顯示前200字，避免輸出過長
                        print("-" * 20)
                else:
                    print("❌ 找不到相關的來源節點。")
                return response.response                      # 回傳最終回答
            except Exception as e:
                return f"❌ 查詢時發生錯誤：{e}"
def main(input_file, idx_path, model_name="gpt-oss:20b", embed_model_name="bge-m3"):
    print(f"🤖 使用 LlamaIndex TreeIndex 的 {model_name} RAG系統測試")
    print(" ==================================================")
    
    # 初始化 RAG 類別
    rag = LlamaIndexTreeRAG(model_name=model_name, embed_model_name=embed_model_name)
    
    # 類比 W10_1 的 folder 儲存檢查機制
    if os.path.exists(idx_path):
        # 1. 如果 folder 存在，直接載入
        rag.load_existing_index(index_dir=idx_path)
    else:
        # 2. 如果 folder 不存在，才新增、切分並建立 treeindex 儲存
        print(f"🔍 儲存資料夾 '{idx_path}' 不存在，正在從原始檔案建立新索引...")
        if not os.path.exists(input_file):
            raise FileNotFoundError(f"❌ 找不到指定的輸入文檔檔案: {input_file}")
            
        # 讀取檔案內容
        with open(input_file, "r", encoding="utf-8") as f:
            content = f.read()
            
        # 🔴 嚴格保留、完全沒刪除的 W10_1/W11_1 專屬切分邏輯
        documents_text = content.split("=== 第")
        documents_text = documents_text[1:]
        documents_text = ["=== 第" + page.strip() for page in documents_text]
        
        # 將切分好的文字封裝回 LlamaIndex 的 Document 物件中
        documents = [Document(text=text) for text in documents_text]
        
        print(f"共讀入 {len(documents)} 頁。")
        for i, doc in enumerate(documents[:5]):  # W11_1 原有的前 5 頁預覽
            snippet = doc.text.replace('\n', ' ')[:10]
            print(f">>> 第{i+1} 頁 <<<\n{snippet}\n")
            
        print("=" * 30)
        # 建立與儲存 TreeIndex
        rag.create_index_from_documents(documents, index_dir=idx_path)
    
    # 列印 TreeIndex 結構與遞迴樹狀結構（W11_1 功能）
    rag.print_tree_index_structure()
    rag.print_tree_structure_recursive(max_depth=3)
    print("=" * 50)
    
    print(f"\n🔍 請輸入查詢問題（輸入 quit 結束）\n", "=" * 60)    
    while True:
        query = input("❓ 問題: ").strip()
        
        if query.lower() == "quit":
            print("\n✅ 結束測試。")
            break
        
        if not query:
            continue
            
        start_time = time.time()
        answer = rag.rag_query(query)
        elapsed_seconds = time.time() - start_time
        
        print(f"🤖 回答: {answer}\n", "-" * 50)
        print(f"⏱️ 查詢花費時間: {elapsed_seconds:.4f} 秒\n", "-" * 50)


if __name__ == "__main__":
    # 配置輸入檔案與 TreeIndex 儲存資料夾路徑
    main(
        input_file="114_碩士班招生簡章_分頁.txt", 
        idx_path="app6_tree_storage",  # 儲存目錄名稱
        model_name="gpt-oss:20b",
        embed_model_name="bge-m3"
    )

🤖 使用 LlamaIndex TreeIndex 的 gpt-oss:20b RAG系統測試
✅ Ollama 服務器 http://localhost:11434 可達。
✅ 成功載入現有索引於 'app6_tree_storage'。

🌳 TreeIndex 架構:
共有 54 個節點。
 列印所有節點的摘要:
--------------------------------------------------
Node ID: 7d2cdee1-76db-4452-bbcb-63bca6218fc6
來源檔案: N/A, 頁碼: N/A
父節點ID: N/A
摘要: === 第1 頁 === 113 年 9 月 19 日本校   招生委員會會議通過                                                                                                           世新大學      114 學年度  碩士班招生簡章                      世新大學招生委員會印製  校址：116 臺北市文山區木柵路一段十七巷一號  網址：https://www.s...
--------------------------------------------------
Node ID: 0a9eaf63-af0a-4c85-8070-3c89e0a13592
來源檔案: N/A, 頁碼: N/A
父節點ID: N/A
摘要: === 第2 頁 === 114 學 年 度 碩 士 班 招 生 重 要 日 程 表  項                目  日        期  簡章網路公告（自行下載）  https://www.shu.edu.tw/招生與入學→招生資訊網→招生 訊息→碩士班  113 年 9 月 26 日起  網路報名  （網路報名作業流程詳見報名流程圖）  113 年 12 月 6 日上午 10:00 起至  114 年 1 月 17 日下午 2:30 止  報名費繳交  （報名費繳費方式說明詳見簡...
--------------------------------------------------
Node ID: 11ed72b7-

❓ 問題:  報名費多少，本校畢業生有打折嗎?


🔍 正在查詢...
